# NYC Collision Studio — Build Website Data

Builds **one** JSON file that the website uses as its sole data source:

```
frontend/public/data/nyc_data.json
```

Reads NYC OpenData's `Motor_Vehicle_Collisions_Crashes` table directly so every row is preserved — the old `cleaned_crashes.csv` pipeline silently dropped ~800k crashes that were missing optional fields (off-street name, cross-street, etc.), so the website previously under-reported by ~33%.

If `Motor_Vehicle_Collisions_Crashes.csv` exists in the project root we read from there; otherwise we download it from NYC OpenData on the fly. The final cell asserts every total matches every breakdown — if it prints `All consistency checks passed`, the website's numbers are guaranteed accurate.

## 1. Imports + paths

Resolve paths relative to the notebook so it works no matter where Jupyter is launched.

In [ ]:
import json
import ssl
from pathlib import Path
from urllib.request import urlopen

import certifi
import numpy as np
import pandas as pd

HERE = Path.cwd()
RAW_PATH = HERE / "Motor_Vehicle_Collisions_Crashes.csv"
OUT_PATH = HERE / "frontend" / "public" / "data" / "nyc_data.json"
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

CRASHES_URL = (
    "https://data.cityofnewyork.us/api/views/h9gi-nx95/rows.csv?accessType=download"
)

print("Raw CSV path :", RAW_PATH)
print("Output JSON  :", OUT_PATH)

## 2. Load the raw NYC OpenData crashes table

Each row in this table is **one crash** — there's no person-level join here, so no dedup is needed. The file is ~600 MB; we only read the columns we need to keep memory reasonable. If the CSV is already in the project root, we use it; otherwise we stream it from NYC OpenData.

In [ ]:
USECOLS = [
    "COLLISION_ID",
    "CRASH DATE",
    "CRASH TIME",
    "BOROUGH",
    "ON STREET NAME",
    "OFF STREET NAME",
    "CROSS STREET NAME",
    "LATITUDE",
    "LONGITUDE",
    "LOCATION",
    "CONTRIBUTING FACTOR VEHICLE 1",
    "VEHICLE TYPE CODE 1",
    "NUMBER OF PERSONS INJURED",
    "NUMBER OF PERSONS KILLED",
    "NUMBER OF PEDESTRIANS INJURED",
    "NUMBER OF PEDESTRIANS KILLED",
    "NUMBER OF CYCLIST INJURED",
    "NUMBER OF CYCLIST KILLED",
    "NUMBER OF MOTORIST INJURED",
    "NUMBER OF MOTORIST KILLED",
]
INT_COLS = [c for c in USECOLS if c.startswith("NUMBER OF")]

if not RAW_PATH.exists():
    print(f"Downloading {CRASHES_URL} to {RAW_PATH.name} ...")
    ctx = ssl.create_default_context(cafile=certifi.where())
    with urlopen(CRASHES_URL, context=ctx, timeout=900) as r, open(RAW_PATH, "wb") as f:
        while True:
            chunk = r.read(1 << 20)
            if not chunk:
                break
            f.write(chunk)
    print("Download finished.")

print("Loading CSV (this can take 1–3 minutes)…")
crashes = pd.read_csv(RAW_PATH, usecols=USECOLS, low_memory=False)
print(f"Loaded {len(crashes):,} crashes.")
assert crashes["COLLISION_ID"].is_unique, "COLLISION_ID should already be unique in the raw crashes table"


## 3. Derive helper columns

- Coerce injury / fatality counts to `int64`.
- Build a unified `CRASH_DATETIME` column out of `CRASH DATE` and `CRASH TIME` (the raw table stores them separately).
- Extract `year` / `hour` for groupby.
- Replace NaN / blank categoricals with `"Unknown"` so groupby keeps every row.

In [ ]:
for c in INT_COLS:
    crashes[c] = pd.to_numeric(crashes[c], errors="coerce").fillna(0).astype("int64")

crashes["CRASH_DATETIME"] = pd.to_datetime(
    crashes["CRASH DATE"].astype(str) + " " + crashes["CRASH TIME"].astype(str),
    errors="coerce",
)
crashes["year"] = crashes["CRASH_DATETIME"].dt.year
crashes["hour"] = crashes["CRASH_DATETIME"].dt.hour
bad = crashes["year"].isna().sum()
print(f"Rows with unparseable datetime (dropped): {bad}")
crashes = crashes.dropna(subset=["year", "hour"]).copy()
crashes["year"] = crashes["year"].astype("int64")
crashes["hour"] = crashes["hour"].astype("int64")

for c in ["BOROUGH", "CONTRIBUTING FACTOR VEHICLE 1", "VEHICLE TYPE CODE 1", "ON STREET NAME"]:
    crashes[c] = crashes[c].fillna("Unknown").astype(str).str.strip()
    crashes.loc[crashes[c] == "", c] = "Unknown"
    # OpenData encodes 'Unspecified' for missing-but-recorded factors — fold into Unknown.
    crashes.loc[crashes[c].str.lower() == "unspecified", c] = "Unknown"

print(
    f"Working set: {len(crashes):,} crashes "
    f"spanning {int(crashes['year'].min())}–{int(crashes['year'].max())}."
)

## 5. The `summarize()` helper

Builds the exact `Summary` shape the frontend already consumes. We reuse this for:

- the full-NYC summary (`idx.all`),
- every single-dim slice (`byBorough[name]`, `byYear[year]`, …),
- every 2-dim join cell (`joins.boroughYear[b][y]`, …).

Same shape everywhere → no schema branching in the frontend → numbers always agree.

In [ ]:
HOURS = list(range(24))

def summarize(df_slice: pd.DataFrame):
    """Aggregate one slice into the Summary dict the website expects.

    Headline totals are computed from the per-victim-type breakdowns so the
    KPI tile (`totalInjured` / `totalKilled`) always agrees with the
    "Who is being harmed" panel. NYC OpenData occasionally has uncategorised
    persons in `NUMBER OF PERSONS INJURED` that don't appear in any
    ped / cyc / mot bucket; using the bucket sum keeps the website internally
    consistent.

    Returns None for empty slices so callers can omit those cells.
    """
    n = len(df_slice)
    if n == 0:
        return None

    boroughs_counts = (
        df_slice.groupby("BOROUGH", observed=True).size().sort_values(ascending=False)
    )
    factor_counts = df_slice["CONTRIBUTING FACTOR VEHICLE 1"].value_counts().head(6)
    hour_counts = df_slice.groupby("hour", observed=True).size().reindex(HOURS, fill_value=0)
    year_counts = df_slice.groupby("year", observed=True).size().sort_index()

    ped_inj = int(df_slice["NUMBER OF PEDESTRIANS INJURED"].sum())
    cyc_inj = int(df_slice["NUMBER OF CYCLIST INJURED"].sum())
    mot_inj = int(df_slice["NUMBER OF MOTORIST INJURED"].sum())
    ped_kil = int(df_slice["NUMBER OF PEDESTRIANS KILLED"].sum())
    cyc_kil = int(df_slice["NUMBER OF CYCLIST KILLED"].sum())
    mot_kil = int(df_slice["NUMBER OF MOTORIST KILLED"].sum())

    return {
        "totalCollisions": int(n),
        "totalInjured":   ped_inj + cyc_inj + mot_inj,
        "totalKilled":    ped_kil + cyc_kil + mot_kil,
        "boroughs": [
            {"name": str(name), "value": int(v)}
            for name, v in boroughs_counts.items()
        ],
        "personTypeBreakdown": [],
        "topFactors": [
            {"name": str(name), "value": int(v)}
            for name, v in factor_counts.items()
        ],
        "collisionsByHourData": [
            {"hour": f"{h}:00", "collisions": int(hour_counts.loc[h])}
            for h in HOURS
        ],
        "crashesByYearData": [
            {"year": int(y), "crashes": int(v)}
            for y, v in year_counts.items()
        ],
        "injured": {
            "pedestrians": ped_inj,
            "cyclists":    cyc_inj,
            "motorists":   mot_inj,
        },
        "killed": {
            "pedestrians": ped_kil,
            "cyclists":    cyc_kil,
            "motorists":   mot_kil,
        },
    }

## 6. Full-dataset summary + filter options

`idx_all` is the canonical full-NYC summary. The hero KPIs and the *Who is being harmed* panel are wired to this number — they never change as chart filters move.

Filter option lists are capped to the most common values so the join indexes stay compact (well under GitHub's 50 MB push limit).

In [ ]:
MAX_FACTORS = 80
MAX_VEHICLES = 60
MAX_STREETS = 50

idx_all = summarize(crashes)
print(
    f"Full dataset: {idx_all['totalCollisions']:,} crashes, "
    f"{idx_all['totalInjured']:,} injured, {idx_all['totalKilled']:,} killed."
)

borough_opts = sorted(
    b for b in crashes["BOROUGH"].unique() if b and b != "Unknown"
)
year_opts = [str(int(y)) for y in sorted(crashes["year"].unique(), reverse=True)]
factor_opts = (
    crashes.loc[crashes["CONTRIBUTING FACTOR VEHICLE 1"] != "Unknown", "CONTRIBUTING FACTOR VEHICLE 1"]
    .value_counts()
    .head(MAX_FACTORS)
    .index.tolist()
)
vehicle_opts = (
    crashes.loc[crashes["VEHICLE TYPE CODE 1"] != "Unknown", "VEHICLE TYPE CODE 1"]
    .value_counts()
    .head(MAX_VEHICLES)
    .index.tolist()
)
street_opts = (
    crashes.loc[crashes["ON STREET NAME"] != "Unknown", "ON STREET NAME"]
    .value_counts()
    .head(MAX_STREETS)
    .index.tolist()
)

print(
    f"Filter options: {len(borough_opts)} boroughs, {len(year_opts)} years, "
    f"{len(factor_opts)} factors, {len(vehicle_opts)} vehicles, {len(street_opts)} streets."
)

## 7. Per-dimension indexes

One Summary per value of each filter dimension. These power any **single-filter** chart view (e.g. *Borough = Brooklyn*, *Year = 2020*, *Fatalities only*).

In [ ]:
def build_index(values, col, cast=None):
    out = {}
    for v in values:
        key = v
        match = cast(v) if cast else v
        s = summarize(crashes[crashes[col] == match])
        if s is not None:
            out[key] = s
    return out

by_borough = build_index(borough_opts, "BOROUGH")
by_year    = build_index(year_opts,    "year", cast=int)
by_factor  = build_index(factor_opts,  "CONTRIBUTING FACTOR VEHICLE 1")
by_vehicle = build_index(vehicle_opts, "VEHICLE TYPE CODE 1")
by_street  = build_index(street_opts,  "ON STREET NAME")

print(
    "1-dim indexes:",
    {
        "byBorough":     len(by_borough),
        "byYear":        len(by_year),
        "byFactor":      len(by_factor),
        "byVehicleType": len(by_vehicle),
        "byOnStreet":    len(by_street),
    },
)

## 8. 2-dimension joins

Pre-aggregate every supported (dimA × dimB) cell. Together with the 1-dim indexes above, these cover **every filter combination** the website lets the user build with at most two non-severity dims plus optional `Injuries only` / `Fatalities only`.

Cells where the intersection is empty are omitted — the frontend treats missing cells as `no data for this combination`.

In [ ]:
def build_join(a_values, a_col, b_values, b_col, a_cast=None, b_cast=None):
    out = {}
    for av in a_values:
        a_match = a_cast(av) if a_cast else av
        sub_a = crashes[crashes[a_col] == a_match]
        if len(sub_a) == 0:
            continue
        inner = {}
        for bv in b_values:
            b_match = b_cast(bv) if b_cast else bv
            s = summarize(sub_a[sub_a[b_col] == b_match])
            if s is not None:
                inner[bv] = s
        if inner:
            out[av] = inner
    return out

joins = {
    "boroughYear":     build_join(borough_opts, "BOROUGH", year_opts, "year", b_cast=int),
    "boroughFactor":   build_join(borough_opts, "BOROUGH", factor_opts, "CONTRIBUTING FACTOR VEHICLE 1"),
    "yearFactor":      build_join(year_opts, "year", factor_opts, "CONTRIBUTING FACTOR VEHICLE 1", a_cast=int),
    "boroughVehicle":  build_join(borough_opts, "BOROUGH", vehicle_opts, "VEHICLE TYPE CODE 1"),
    "yearVehicle":     build_join(year_opts, "year", vehicle_opts, "VEHICLE TYPE CODE 1", a_cast=int),
    "boroughOnStreet": build_join(borough_opts, "BOROUGH", street_opts, "ON STREET NAME"),
    "yearOnStreet":    build_join(year_opts, "year", street_opts, "ON STREET NAME", a_cast=int),
}

print("2-dim joins:", {k: sum(len(v) for v in d.values()) for k, d in joins.items()})

## 9. Sample rows for the record-level data table

The website still shows real crash records on the *Record-level explorer* page. We embed a uniform 10 000-row sample so the table works without needing to ship the 2 GB CSV.

Keys here match `DataTable.tsx` (`PREFERRED_COLS`) so React renders them in the right order.

In [ ]:
N_SAMPLE = 10_000
sample_df = crashes.sample(n=min(N_SAMPLE, len(crashes)), random_state=42)

def to_row(r):
    def opt(v):
        return None if pd.isna(v) else v
    return {
        "CRASH_ID":                       str(r["COLLISION_ID"]),
        "CRASH_DATETIME":                 r["CRASH_DATETIME"],
        "BOROUGH":                        r["BOROUGH"],
        "ON_STREET_NAME":                 r["ON STREET NAME"],
        "OFF_STREET_NAME":                opt(r["OFF STREET NAME"]),
        "CROSS_STREET_NAME":              opt(r["CROSS STREET NAME"]),
        "LATITUDE":                       opt(r["LATITUDE"]),
        "LONGITUDE":                      opt(r["LONGITUDE"]),
        "LOCATION":                       opt(r["LOCATION"]),
        "PERSON_TYPE":                    opt(r["PERSON_TYPE"]),
        "CONTRIBUTING FACTOR VEHICLE 1":  r["CONTRIBUTING FACTOR VEHICLE 1"],
        "VEHICLE TYPE CODE 1":            r["VEHICLE TYPE CODE 1"],
        "NUMBER OF PERSONS INJURED":      int(r["NUMBER OF PERSONS INJURED"]),
        "NUMBER OF PERSONS KILLED":       int(r["NUMBER OF PERSONS KILLED"]),
        "NUMBER OF PEDESTRIANS INJURED":  int(r["NUMBER OF PEDESTRIANS INJURED"]),
        "NUMBER OF PEDESTRIANS KILLED":   int(r["NUMBER OF PEDESTRIANS KILLED"]),
        "NUMBER OF CYCLIST INJURED":      int(r["NUMBER OF CYCLIST INJURED"]),
        "NUMBER OF CYCLIST KILLED":       int(r["NUMBER OF CYCLIST KILLED"]),
        "NUMBER OF MOTORIST INJURED":     int(r["NUMBER OF MOTORIST INJURED"]),
        "NUMBER OF MOTORIST KILLED":      int(r["NUMBER OF MOTORIST KILLED"]),
    }

sample_rows = [to_row(r) for _, r in sample_df.iterrows()]
print(f"Built {len(sample_rows):,} sample rows for the data table.")

## 10. Assemble + write `nyc_data.json`

Everything the website needs in **one** file. The four top-level keys map one-to-one onto the existing TypeScript types — minimal frontend churn.

- `datasetMetadata` — full-NYC totals (hero KPIs, *Who is being harmed*, *Full dataset report*).
- `datasetIndex` — pre-aggregated slices for filtered chart views (single-dim indexes + 2-dim joins).
- `filterOptions` — values that populate the filter dropdowns.
- `sampleRows` — 10k crash rows for the *Record-level explorer* table.

In [ ]:
hour_counts_full = crashes.groupby("hour").size().reindex(HOURS, fill_value=0)
year_counts_full = crashes.groupby("year").size().sort_index()
factor_counts_full = (
    crashes.loc[crashes["CONTRIBUTING FACTOR VEHICLE 1"] != "Unknown",
                "CONTRIBUTING FACTOR VEHICLE 1"]
    .value_counts()
    .head(10)
)

dataset_metadata = {
    "total_collisions":         idx_all["totalCollisions"],
    "total_injured":            idx_all["totalInjured"],
    "total_killed":             idx_all["totalKilled"],
    "boroughs": {
        b["name"]: b["value"] for b in idx_all["boroughs"] if b["name"] != "Unknown"
    },
    "crashes_by_hour":          {str(h): int(hour_counts_full.loc[h]) for h in HOURS},
    "injuries_by_type": {
        "pedestrians_injured": idx_all["injured"]["pedestrians"],
        "cyclists_injured":    idx_all["injured"]["cyclists"],
        "motorists_injured":   idx_all["injured"]["motorists"],
    },
    "fatalities_by_type": {
        "pedestrians_killed": idx_all["killed"]["pedestrians"],
        "cyclists_killed":    idx_all["killed"]["cyclists"],
        "motorists_killed":   idx_all["killed"]["motorists"],
    },
    "top_contributing_factors": {str(n): int(v) for n, v in factor_counts_full.items()},
    "crashes_by_year":          {str(int(y)): int(c) for y, c in year_counts_full.items()},
}

dataset_index = {
    "all":           idx_all,
    "byBorough":     by_borough,
    "byYear":        by_year,
    "byFactor":      by_factor,
    "byVehicleType": by_vehicle,
    "byOnStreet":    by_street,
    "joins":         joins,
}

filter_options = {
    "boroughs":     borough_opts,
    "factors":      factor_opts,
    "vehicleTypes": vehicle_opts,
    "onStreets":    street_opts,
    "years":        year_opts,
}

out = {
    "version":         1,
    "generatedAt":     pd.Timestamp.now("UTC").isoformat(),
    "datasetMetadata": dataset_metadata,
    "datasetIndex":    dataset_index,
    "filterOptions":   filter_options,
    "sampleRows":      sample_rows,
}

OUT_PATH.write_text(json.dumps(out, separators=(",", ":")))
size_mb = OUT_PATH.stat().st_size / (1024 * 1024)
print(f"Wrote {OUT_PATH} ({size_mb:.1f} MB)")

## 11. Consistency assertions

Every total must agree with every breakdown that derives from it. If this cell raises, the website would display inconsistent numbers — re-run the notebook before deploying.

In [ ]:
m = dataset_metadata
a = dataset_index["all"]

assert m["total_collisions"] == a["totalCollisions"], "metadata vs index disagree on total_collisions"
assert m["total_injured"]    == a["totalInjured"],    "metadata vs index disagree on total_injured"
assert m["total_killed"]     == a["totalKilled"],     "metadata vs index disagree on total_killed"

assert sum(m["crashes_by_year"].values()) == m["total_collisions"], \
    "year buckets don't sum to total"
assert sum(m["injuries_by_type"].values()) == m["total_injured"], \
    "injury breakdown doesn't sum to total_injured"
assert sum(m["fatalities_by_type"].values()) == m["total_killed"], \
    "fatality breakdown doesn't sum to total_killed"

per_year_killed = sum(by_year[y]["totalKilled"] for y in by_year)
assert per_year_killed == m["total_killed"], \
    f"byYear killed sum {per_year_killed} != {m['total_killed']}"

# borough-known killed should never exceed total (Unknown borough is excluded)
per_borough_killed_known = sum(by_borough[b]["totalKilled"] for b in by_borough)
assert per_borough_killed_known <= m["total_killed"], \
    "borough-known killed exceeds total"

print("All consistency checks passed.")
print(f"  Total crashes: {m['total_collisions']:,}")
print(f"  Total injured: {m['total_injured']:,}")
print(f"  Total killed : {m['total_killed']:,}")
print(f"  Years        : {min(m['crashes_by_year'])}–{max(m['crashes_by_year'])}")
print(f"  Sample rows  : {len(sample_rows):,}")
print(f"  Output size  : {size_mb:.1f} MB")